In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from ucimlrepo import fetch_ucirepo
from sklearn.metrics import roc_auc_score, f1_score, recall_score, accuracy_score

In [2]:
def load_data():
    parkinsons = fetch_ucirepo(id=174)
    df = parkinsons.data.original.copy()
    df["subject"] = df["name"].str.extract(r"(R\d+_S\d+)")[0]
    return df

In [3]:
def get_features(df):
    return [c for c in df.columns if c not in ("name", "status", "subject")]

In [4]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden=64, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.BatchNorm1d(hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

In [5]:
class ResidualMLP(nn.Module):
    def __init__(self, input_dim, hidden=64, dropout=0.3):
        super().__init__()
        self.in_layer = nn.Linear(input_dim, hidden)
        self.block1 = nn.Sequential(
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
        )
        self.block2 = nn.Sequential(
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
        )
        self.out = nn.Linear(hidden, 1)

    def forward(self, x):
        h = self.in_layer(x)
        h = h + self.block1(h)
        h = h + self.block2(h)
        return self.out(h).squeeze(-1)

In [6]:
def build_model(model_type, input_dim):
    if model_type == "residual":
        return ResidualMLP(input_dim=input_dim)
    return MLP(input_dim=input_dim)

In [7]:
def load_artifact(path):
    return torch.load(path, map_location="cpu", weights_only=False)

In [8]:
def evaluate_artifact(artifact, df):
    features = artifact["feature_names"]
    X = df[features].values
    y = df["status"].values.astype(int)

    mean = np.array(artifact["scaler_mean"], dtype=np.float32)
    scale = np.array(artifact["scaler_scale"], dtype=np.float32)
    X_scaled = (X - mean) / scale

    model = build_model(artifact["model_type"], artifact["input_dim"])
    model.load_state_dict(artifact["model_state_dict"])
    model.eval()

    with torch.no_grad():
        logits = model(torch.tensor(X_scaled, dtype=torch.float32))
        probs = torch.sigmoid(logits).numpy()

    threshold = artifact["threshold"]
    preds = (probs >= threshold).astype(int)

    auc = roc_auc_score(y, probs)
    f1 = f1_score(y, preds)
    rec = recall_score(y, preds)
    acc = accuracy_score(y, preds)
    return {"auc": auc, "f1": f1, "recall": rec, "acc": acc, "threshold": threshold}

In [9]:
def resolve_experiment_dir(experiment_dir):
    candidates = [
        experiment_dir,
        os.path.join("files", experiment_dir),
        os.path.join("..", experiment_dir),
        os.path.join("..", "files", experiment_dir),
    ]
    for c in candidates:
        if os.path.isdir(c):
            return c
    raise FileNotFoundError(f"Could not find directory for: {experiment_dir}")


def run_saved_experiment(experiment_dir, df):
    experiment_dir = resolve_experiment_dir(experiment_dir)
    rows = []
    for fold in range(1, 6):
        artifact_path = os.path.join(experiment_dir, f"fold_{fold}.pt")
        artifact = load_artifact(artifact_path)
        m = evaluate_artifact(artifact, df)
        m["fold"] = fold
        rows.append(m)

    table = pd.DataFrame(rows)
    avg = table[["auc", "f1", "recall", "acc"]].mean().to_dict()
    return table, avg

In [10]:
df = load_data()
print("cwd:", os.getcwd())
print(df.shape)
print(df["status"].value_counts().to_dict())

cwd: /Users/jichuan/Desktop/273/CS273P-Final-Project/files
(195, 25)
{1: 147, 0: 48}


In [11]:
baseline_table, baseline_avg = run_saved_experiment("saved_models/baseline_mlp_bce", df)
print("Baseline fold results")
display(baseline_table)
print("Baseline avg:", baseline_avg)

Baseline fold results


,auc,f1,recall,acc,threshold,fold
0,0.938209,0.710526,0.551020,0.661538,0.5,1
1,0.911990,0.765432,0.632653,0.707692,0.5,2
2,0.920210,0.818533,0.721088,0.758974,0.5,3
3,0.864512,0.855172,0.843537,0.784615,0.5,4
4,0.908588,0.760331,0.625850,0.702564,0.5,5


Baseline avg: {'auc': 0.9087018140589569, 'f1': 0.7819988450786449, 'recall': 0.6748299319727892, 'acc': 0.723076923076923}


In [12]:
focal_table, focal_avg = run_saved_experiment("saved_models/residual_mlp_focal", df)
print("Residual + Focal fold results")
display(focal_table)
print("Residual + Focal avg:", focal_avg)

Residual + Focal fold results


,auc,f1,recall,acc,threshold,fold
0,0.897109,0.893204,0.938776,0.830769,0.5,1
1,0.852041,0.839590,0.836735,0.758974,0.5,2
2,0.830499,0.839344,0.870748,0.748718,0.5,3
3,0.904053,0.941176,0.979592,0.907692,0.5,4
4,0.893991,0.883117,0.925170,0.815385,0.5,5


Residual + Focal avg: {'auc': 0.8755385487528345, 'f1': 0.8792863886362706, 'recall': 0.9102040816326531, 'acc': 0.8123076923076923}


In [13]:
hybrid_table, hybrid_avg = run_saved_experiment("saved_models/residual_mlp_hybrid", df)
print("Residual + Hybrid fold results")
display(hybrid_table)
print("Residual + Hybrid avg:", hybrid_avg)

Residual + Hybrid fold results


,auc,f1,recall,acc,threshold,fold
0,0.821429,0.873846,0.965986,0.789744,0.36,1
1,0.836310,0.801498,0.727891,0.728205,0.46,2
2,0.886338,0.884013,0.959184,0.810256,0.32,3
3,0.869189,0.862191,0.829932,0.800000,0.40,4
4,0.892149,0.862319,0.809524,0.805128,0.52,5


Residual + Hybrid avg: {'auc': 0.8610827664399092, 'f1': 0.8567732947344977, 'recall': 0.8585034013605443, 'acc': 0.7866666666666665}


In [14]:
final_summary = pd.DataFrame([
    {"experiment": "baseline_mlp_bce", **baseline_avg},
    {"experiment": "residual_mlp_focal", **focal_avg},
    {"experiment": "residual_mlp_hybrid", **hybrid_avg},
]).sort_values("auc", ascending=False)

print("Final comparison from saved models")
display(final_summary)

Final comparison from saved models


,experiment,auc,f1,recall,acc
0,baseline_mlp_bce,0.908702,0.781999,0.674830,0.723077
1,residual_mlp_focal,0.875539,0.879286,0.910204,0.812308
2,residual_mlp_hybrid,0.861083,0.856773,0.858503,0.786667
